# Real-data temperature and soil-moisture QA

This notebook checks the environmental data downloaded with the Chicken Bone Meadow dataset. It prepares the pieces needed for temperature correction and later comparison between ERT-derived water-content change and soil-moisture sensor change.


In [ ]:
from __future__ import annotations

import re
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'axes.labelsize': 13,
    'axes.titlesize': 13,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 9,
    'figure.dpi': 120,
})

np.set_printoptions(precision=4, suppress=True)


In [ ]:
# Resolve paths.
here = Path.cwd().resolve()
candidates = [here, *here.parents]
project_root = next((p for p in candidates if (p / 'deepert').exists() and (p / 'ProcessedData').exists()), None)
if project_root is None:
    raise FileNotFoundError('Cannot locate project root containing deepert/ and ProcessedData/.')

data_dir = project_root / 'ProcessedData'
result_dir = project_root / 'result' / '8_real_data' / '2_timelapse_inversion_real_deepert'
if not result_dir.exists():
    raise FileNotFoundError(f'Missing time-lapse inversion result directory: {result_dir}')

save_figures = True
figure_dpi = 600
analysis_start = pd.Timestamp('2022-03-26 00:00:00')
analysis_end = pd.Timestamp('2022-05-16 00:00:00')
baseline_time = pd.Timestamp('2022-03-26 00:30:00')

tmc_colors = {
    'TMC1': '#0072B2',
    'TMC2': '#E69F00',
    'TMC4': '#009E73',
    'TMC5': '#D55E00',
    'TMC6': '#CC79A7',
}

def savefig(fig, filename):
    if save_figures:
        out = result_dir / filename
        fig.savefig(out, dpi=figure_dpi, bbox_inches='tight')
        print('saved:', out)

print('project_root =', project_root)
print('data_dir     =', data_dir)
print('result_dir   =', result_dir)


## 1) Sensor locations projected onto the ERT profile


In [ ]:
# Load ERT geometry from the inversion result and project sensor coordinates to the ERT profile.
locations = pd.read_csv(data_dir / 'Locations.csv')
with np.load(result_dir / 'real_data_geometry_and_survey.npz') as geom:
    elec_x = np.asarray(geom['elec_x'], dtype=float)
    elec_z = np.asarray(geom['elec_z'], dtype=float)
    elevation_reference = float(np.nanmax(np.asarray(geom['elec_elevation'], dtype=float)))

start_row = locations.loc[locations['ID'].eq('ERT-Start')].iloc[0]
end_row = locations.loc[locations['ID'].eq('ERT-End')].iloc[0]
start_xy = np.array([start_row['Easting'], start_row['Northing']], dtype=float)
end_xy = np.array([end_row['Easting'], end_row['Northing']], dtype=float)
profile_vec = end_xy - start_xy
profile_len_map = float(np.linalg.norm(profile_vec))
profile_len_ert = float(np.nanmax(elec_x) - np.nanmin(elec_x))

def project_to_profile(row):
    xy = np.array([row['Easting'], row['Northing']], dtype=float)
    fraction = float(np.dot(xy - start_xy, profile_vec) / np.dot(profile_vec, profile_vec))
    return pd.Series({
        'profile_fraction': fraction,
        'profile_x_m': float(np.nanmin(elec_x) + fraction * profile_len_ert),
        'profile_horizontal_m': fraction * profile_len_map,
        'relative_elevation_m': float(row['Elevation'] - elevation_reference),
    })

locations_proj = pd.concat([locations, locations.apply(project_to_profile, axis=1)], axis=1)
sensor_locations = locations_proj[locations_proj['ID'].str.startswith('TMC')].copy()
sensor_locations = sensor_locations.sort_values('profile_x_m')
sensor_locations.to_csv(result_dir / 'real_sensor_locations_projected.csv', index=False)

fig, ax = plt.subplots(figsize=(11.0, 3.6))
ax.plot(elec_x, elec_z, color='black', lw=1.4, label='ERT topography')
for _, row in sensor_locations.iterrows():
    color = tmc_colors.get(row['ID'], 'tab:gray')
    ax.scatter(row['profile_x_m'], row['relative_elevation_m'], s=60, color=color, edgecolor='black', zorder=4)
    ax.text(row['profile_x_m'], row['relative_elevation_m'] + 2.0, row['ID'], ha='center', va='bottom', fontsize=10)
ax.set_xlim(float(elec_x.min()), float(elec_x.max()))
ax.set_ylim(min(float(elec_z.min()), float(sensor_locations['relative_elevation_m'].min())) - 5.0, 5.0)
ax.set_aspect('equal', adjustable='box')
ax.set_xlabel('ERT profile distance (m)')
ax.set_ylabel('Relative elevation (m)')
ax.set_title('TMC sensor locations projected onto the ERT profile')
ax.grid(alpha=0.25)
fig.tight_layout()
savefig(fig, 'real_sensor_locations_projected.png')
plt.show()

sensor_locations[['ID', 'profile_x_m', 'relative_elevation_m', 'Type']]


## 2) Load environmental time series


In [ ]:
def read_units_csv(path: Path, time_col: str) -> pd.DataFrame:
    df = pd.read_csv(path, skiprows=[1], low_memory=False)
    df[time_col] = pd.to_datetime(df[time_col])
    df = df.set_index(time_col).sort_index()
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df = df.replace(-9999, np.nan)
    return df

soil_dir = data_dir / 'Soil_Moisture_Data'
temp_dir = data_dir / 'Soil_Temperature_Data'
soil = {path.stem: read_units_csv(path, 'DateTime') for path in sorted(soil_dir.glob('TMC*.csv'))}
dtp = {path.stem.replace('DTP_', ''): read_units_csv(path, 'DateTime') for path in sorted(temp_dir.glob('DTP_TMC*.csv'))}
weather = read_units_csv(data_dir / 'Weather_Well_Data.csv', 'Date')

used_lines = (result_dir / 'used_data_files.txt').read_text().splitlines()
ert_times = pd.to_datetime([line.split('	')[0] for line in used_lines])
ert_days = (ert_times - ert_times[0]).total_seconds() / 86400.0

print('Soil moisture stations:', sorted(soil))
print('DTP temperature stations:', sorted(dtp))
print('ERT time range:', ert_times[0], 'to', ert_times[-1], 'n=', len(ert_times))
print('Weather range:', weather.index.min(), 'to', weather.index.max())


In [ ]:
# Interpolate hourly soil-moisture/sensor-temperature series to the exact ERT acquisition times.
sensor_records = []
for station, df in soil.items():
    station_meta = sensor_locations.loc[sensor_locations['ID'].eq(station)]
    profile_x = float(station_meta['profile_x_m'].iloc[0]) if not station_meta.empty else np.nan
    merged_index = df.index.union(ert_times)
    interp = df.reindex(merged_index).interpolate(method='time').reindex(ert_times)
    for timestamp, row in interp.iterrows():
        out = {'station': station, 'DateTime': timestamp, 'day_since_baseline': (timestamp - ert_times[0]).total_seconds() / 86400.0, 'profile_x_m': profile_x}
        for col in interp.columns:
            out[col] = row[col]
        sensor_records.append(out)
sensor_at_ert = pd.DataFrame(sensor_records)
sensor_at_ert.to_csv(result_dir / 'real_sensor_timeseries_at_ert_times.csv', index=False)
print('saved:', result_dir / 'real_sensor_timeseries_at_ert_times.csv')
sensor_at_ert.head()


## 3) Soil-moisture changes during the snowmelt window


In [ ]:
depths = [0.1, 0.3, 0.6, 0.8]
fig, axes = plt.subplots(2, 2, figsize=(13.5, 8.0), sharex=True)
axes = axes.ravel()

for ax, depth in zip(axes, depths, strict=True):
    col = f'MC_{depth:.1f}m'
    for station in sorted(soil):
        df = soil[station]
        if col not in df.columns:
            continue
        window = df.loc[analysis_start:analysis_end, col]
        ref_index = df.index.union(pd.DatetimeIndex([baseline_time]))
        ref_series = df[col].reindex(ref_index).interpolate(method='time')
        reference = float(ref_series.loc[baseline_time])
        ax.plot(window.index, window - reference, lw=1.5, color=tmc_colors.get(station), label=station)
    ax.axhline(0.0, color='black', lw=0.8, alpha=0.7)
    ax.set_title(f'Depth {depth:.1f} m')
    ax.set_ylabel('Delta theta (V/V)')
    ax.grid(alpha=0.25)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))

axes[-2].set_xlabel('Date in 2022')
axes[-1].set_xlabel('Date in 2022')
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=len(labels), frameon=True, bbox_to_anchor=(0.5, 1.02))
fig.suptitle('Soil-moisture change relative to 2022-03-26 00:30', y=1.06)
fig.tight_layout()
savefig(fig, 'real_soil_moisture_delta_QA.png')
plt.show()


In [ ]:
# Profile view of sensor Delta theta at the ERT acquisition times.
fig, axes = plt.subplots(2, 2, figsize=(13.5, 7.8), sharex=True, sharey=True)
axes = axes.ravel()
for ax, depth in zip(axes, depths, strict=True):
    col = f'MC_{depth:.1f}m'
    table = sensor_at_ert.pivot(index='DateTime', columns='station', values=col)
    x_lookup = sensor_at_ert.drop_duplicates('station').set_index('station')['profile_x_m']
    stations = [s for s in sorted(table.columns) if np.isfinite(x_lookup.get(s, np.nan))]
    x_vals = np.array([x_lookup[s] for s in stations], dtype=float)
    values = table[stations].to_numpy(dtype=float)
    ref = values[0:1, :]
    delta = values - ref
    for time_idx in [0, len(ert_times)//3, 2*len(ert_times)//3, len(ert_times)-1]:
        ax.plot(x_vals, delta[time_idx], marker='o', lw=1.6, label=ert_times[time_idx].strftime('%m-%d'))
    ax.axhline(0.0, color='black', lw=0.8, alpha=0.7)
    ax.set_title(f'Depth {depth:.1f} m')
    ax.set_xlabel('ERT profile distance (m)')
    ax.set_ylabel('Delta theta (V/V)')
    ax.grid(alpha=0.25)
axes[0].legend(frameon=True, ncol=2)
fig.tight_layout()
savefig(fig, 'real_soil_moisture_profile_snapshots.png')
plt.show()


## 4) Soil and snow temperature checks

DTP columns are named by sensor position in centimeters. Negative positions are below ground and are the most relevant for ERT temperature correction.


In [ ]:
def parse_temp_position_cm(column: str) -> float | None:
    match = re.match(r'temp_(-?\d+(?:\.\d+)?)$', column)
    if match is None:
        return None
    return float(match.group(1))

def underground_temperature_matrix(df: pd.DataFrame):
    pairs = []
    for col in df.columns:
        pos_cm = parse_temp_position_cm(col)
        if pos_cm is not None and pos_cm < 0:
            pairs.append((abs(pos_cm) / 100.0, col))
    pairs = sorted(pairs)
    depths_m = np.array([depth for depth, _ in pairs], dtype=float)
    cols = [col for _, col in pairs]
    return depths_m, df[cols]

fig, axes = plt.subplots(2, 2, figsize=(13.5, 8.2), sharex=True, sharey=True)
axes = axes.ravel()
last_mesh = None
for ax, station in zip(axes, sorted(dtp), strict=False):
    df = dtp[station].loc[analysis_start:analysis_end].resample('6h').mean()
    depths_m, temp_table = underground_temperature_matrix(df)
    if depths_m.size == 0:
        ax.set_title(f'{station}: no underground DTP depths')
        continue
    temp_values = temp_table.to_numpy(dtype=float).T
    t_num = mdates.date2num(df.index.to_pydatetime())
    last_mesh = ax.pcolormesh(t_num, depths_m, temp_values, shading='auto', cmap='coolwarm', vmin=-2.0, vmax=8.0)
    try:
        ax.contour(t_num, depths_m, temp_values, levels=[0.0], colors='black', linewidths=0.8)
    except Exception:
        pass
    ax.set_title(station)
    ax.invert_yaxis()
    ax.set_ylabel('Depth below ground (m)')
    ax.xaxis_date()
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
    ax.grid(False)
for ax in axes[-2:]:
    ax.set_xlabel('Date in 2022')
if last_mesh is not None:
    cbar = fig.colorbar(last_mesh, ax=axes, fraction=0.018, pad=0.02)
    cbar.set_label('Temperature (deg C)')
fig.suptitle('Underground DTP temperature profiles during the snowmelt window', y=1.02)
fig.tight_layout()
savefig(fig, 'real_soil_temperature_profiles_QA.png')
plt.show()


In [ ]:
# TEROS temperatures at the same depths as the moisture sensors.
fig, axes = plt.subplots(2, 2, figsize=(13.5, 8.0), sharex=True)
axes = axes.ravel()
for ax, depth in zip(axes, depths, strict=True):
    col = f'T_{depth:.1f}m'
    for station in sorted(soil):
        df = soil[station]
        if col not in df.columns:
            continue
        window = df.loc[analysis_start:analysis_end, col]
        ax.plot(window.index, window, lw=1.4, color=tmc_colors.get(station), label=station)
    ax.axhline(0.0, color='black', lw=0.8, alpha=0.7)
    ax.set_title(f'Depth {depth:.1f} m')
    ax.set_ylabel('Temperature (deg C)')
    ax.grid(alpha=0.25)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
axes[-2].set_xlabel('Date in 2022')
axes[-1].set_xlabel('Date in 2022')
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=len(labels), frameon=True, bbox_to_anchor=(0.5, 1.02))
fig.suptitle('TEROS soil-temperature time series during the snowmelt window', y=1.06)
fig.tight_layout()
savefig(fig, 'real_teros_temperature_QA.png')
plt.show()


## 5) Weather, snow and groundwater context


In [ ]:
weather_window = weather.loc[analysis_start.date():analysis_end.date()].copy()
fig, axes = plt.subplots(3, 1, figsize=(12.5, 8.2), sharex=True)

ax = axes[0]
ax.bar(weather_window.index, weather_window['Precipitation'], color='#8DBEFF', width=0.8, label='Precipitation')
ax.set_ylabel('Precipitation (mm/day)')
ax.legend(frameon=True, loc='upper left')
ax.grid(alpha=0.25)

ax = axes[1]
ax.plot(weather_window.index, weather_window['Snow_depth_sonic'], color='#0072B2', lw=1.8, label='Snow depth')
ax.set_ylabel('Snow depth (m)')
ax.legend(frameon=True, loc='upper right')
ax.grid(alpha=0.25)

ax = axes[2]
ax.plot(weather_window.index, weather_window['Air_temp'], color='#D55E00', lw=1.6, label='Air temp')
if 'Water_temp' in weather_window.columns:
    ax.plot(weather_window.index, weather_window['Water_temp'], color='#009E73', lw=1.6, label='Water temp')
ax.axhline(0.0, color='black', lw=0.8, alpha=0.7)
ax.set_ylabel('Temperature (deg C)')
ax.set_xlabel('Date in 2022')
ax.legend(frameon=True, loc='best')
ax.grid(alpha=0.25)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))

fig.suptitle('Weather and well context for the selected ERT snowmelt window', y=1.02)
fig.tight_layout()
savefig(fig, 'real_weather_snow_QA.png')
plt.show()


## Next step

Use the DTP/TEROS temperature data to build a fixed gridded temperature field on the ERT inversion mesh. Then apply the WRR-style correction to convert raw inverted resistivity to temperature-corrected resistivity before interpreting Delta rho / rho0 as hydrologic change.
